# 🧪 AiStock 核心计算引擎测试

## 测试目标
- ✅ TechnicalCalculator: 技术面计算
- ✅ FundamentalCalculator: 基本面评分
- ✅ MacroCalculator: 宏观面联动
- ✅ DynamicPriceEngine: 三维综合计算

## 测试内容
- 技术指标计算准确性
- 基本面评分合理性
- 宏观联动系数
- 动态价格生成

In [ ]:
# 环境设置
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成")

In [ ]:
# 导入核心模块
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from dynamic_price_system.core.technical_calculator import TechnicalCalculator
from dynamic_price_system.core.fundamental_calculator import FundamentalCalculator
from dynamic_price_system.core.macro_calculator import MacroCalculator
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine
from dynamic_price_system.core.price_validator import PriceValidator

print("✅ 核心模块导入成功")

## 1️⃣ 技术面计算测试

In [ ]:
# 生成模拟股票数据
dates = pd.date_range('2025-01-01', periods=200, freq='D')
np.random.seed(42)

close_prices = 40 + np.cumsum(np.random.randn(200) * 0.5)
df = pd.DataFrame({
    'datetime': dates,
    'open': close_prices * (1 + np.random.randn(200) * 0.01),
    'high': close_prices * (1 + np.random.randn(200) * 0.02 + 0.01),
    'low': close_prices * (1 + np.random.randn(200) * 0.02 - 0.01),
    'close': close_prices,
    'volume': np.random.randint(1000000, 5000000, 200)
})

print(f"✅ 生成模拟数据：{len(df)}条")
print(f"📊 最新收盘价：{df['close'].iloc[-1]:.2f}")

In [ ]:
# 测试技术指标计算
tech_calc = TechnicalCalculator(df)

indicators = tech_calc.get_latest_indicators()

print("\n📊 技术指标：")
for k, v in indicators.items():
    if v is not None:
        print(f"  • {k}: {v:.2f}" if isinstance(v, float) else f"  • {k}: {v}")

In [ ]:
# 测试技术面价格计算
entry_price = tech_calc.get_entry_price()
stop_loss = tech_calc.get_stop_loss()
target_price = tech_calc.get_target_price()

print("\n📊 技术面价格：")
print(f"  • 入场价：{entry_price:.2f}")
print(f"  • 止损价：{stop_loss:.2f}")
print(f"  • 目标价：{target_price:.2f}")

current_price = df['close'].iloc[-1]
print(f"\n📊 盈亏比：{(target_price - entry_price) / (entry_price - stop_loss):.2f}")

In [ ]:
# 可视化技术指标
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 收盘价 + 均线
axes[0].plot(df['datetime'], df['close'], label='收盘价', linewidth=2)
axes[0].plot(df['datetime'], df['ma_20'], label='MA20', linestyle='--')
axes[0].plot(df['datetime'], df['ma_60'], label='MA60', linestyle='--')
axes[0].set_title('收盘价与均线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df['datetime'], df['rsi_14'], label='RSI(14)', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='超买线')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='超卖线')
axes[1].set_title('RSI 指标')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 波动率
axes[2].plot(df['datetime'], df['volatility_20'], label='20 日波动率', color='orange')
axes[2].set_title('波动率')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ 技术指标可视化完成")

## 2️⃣ 基本面评分测试

In [ ]:
# 测试不同财务数据的基本面评分
test_cases = [
    {
        'name': '优秀公司',
        'data': {'revenue_growth': 25, 'profit_growth': 30, 'roe': 20, 'gross_margin': 35, 'debt_ratio': 30}
    },
    {
        'name': '良好公司',
        'data': {'revenue_growth': 15, 'profit_growth': 18, 'roe': 12, 'gross_margin': 25, 'debt_ratio': 50}
    },
    {
        'name': '一般公司',
        'data': {'revenue_growth': 5, 'profit_growth': 8, 'roe': 8, 'gross_margin': 15, 'debt_ratio': 65}
    },
    {
        'name': '警惕公司',
        'data': {'revenue_growth': -5, 'profit_growth': -10, 'roe': 3, 'gross_margin': 8, 'debt_ratio': 80}
    }
]

print("📊 基本面评分测试：")
for case in test_cases:
    calc = FundamentalCalculator(case['data'])
    score = calc.get_score()
    factor = calc.get_adjustment_factor()
    rec = calc.get_recommendation()
    
    print(f"\n  {case['name']}:")
    print(f"    • 评分：{score}分")
    print(f"    • 调整系数：{factor:.3f}")
    print(f"    • 建议：{rec}")

## 3️⃣ 宏观面联动测试

In [ ]:
# 测试宏观面计算
config = ConfigService(system_name='dynamic_price')

# 模拟宏观数据
macro_data = {
    'brent_crude': 104.66,
    'comex_gold': 4693.38,
    'lme_copper': 9500,
    'pmi': 51.2,
    'm2_growth': 9.5,
    'usd_cny': 7.22
}

print("\n📊 宏观面系数测试：")
sectors = ['油气开采', '黄金', '新能源', '特高压', '军工']

for sector in sectors:
    calc = MacroCalculator(macro_data, sector=sector, config=config)
    factor = calc.get_adjustment_factor()
    print(f"  • {sector}: {factor:.3f}")

## 4️⃣ 三维综合计算测试

In [ ]:
# 测试完整的价格引擎
cache = CacheService(max_size=1000, ttl=3600)
engine = DynamicPriceEngine(config_service=config, cache_service=cache)

# 准备测试数据
stocks_data = {'600938': df}
financial_data = {'600938': {'revenue_growth': 15, 'profit_growth': 20, 'roe': 18, 'gross_margin': 30, 'debt_ratio': 40}}

results = engine.calculate_all(
    stocks_data=stocks_data,
    financial_data=financial_data,
    macro_data=macro_data
)

print("\n📊 三维综合计算结果：")
for result in results:
    print(f"\n  {result['code']} {result.get('sector', '')}:")
    print(f"    • 当前价：{result['current_price']:.2f}")
    print(f"    • 入场价：{result['entry_price']:.2f}")
    print(f"    • 止损价：{result['stop_loss']:.2f}")
    print(f"    • 目标价：{result['target_price']:.2f}")
    print(f"    • 盈亏比：{result['profit_loss_ratio']:.2f}")
    print(f"    • 基本面评分：{result['fundamental_score']}")
    print(f"    • 宏观系数：{result['macro_factor']:.3f}")
    print(f"    • 建议：{result['recommendation']}")

## 5️⃣ 价格验证测试

In [ ]:
# 测试价格验证器
validator = PriceValidator(config)

test_prices = [
    {'entry': 40, 'stop': 36, 'target': 48, 'current': 42},
    {'entry': 40, 'stop': 42, 'target': 48, 'current': 42},  # 止损高于入场
    {'entry': 40, 'stop': 38, 'target': 41, 'current': 42},  # 盈亏比过低
]

print("\n📊 价格验证测试：")
for i, test in enumerate(test_prices, 1):
    result = validator.validate_all(
        test['entry'], test['stop'], test['target'], test['current']
    )
    
    print(f"\n  测试{i}:")
    print(f"    • 入场价：{test['entry']} → {result['entry_price']}")
    print(f"    • 止损价：{test['stop']} → {result['stop_loss']}")
    print(f"    • 目标价：{test['target']} → {result['target_price']}")
    print(f"    • 盈亏比：{result['profit_loss_ratio']:.2f}")
    print(f"    • 有效性：{result['is_valid']}")

## 📊 测试总结

In [ ]:
print("="*60)
print("📋 核心计算引擎测试总结")
print("="*60)
print("✅ TechnicalCalculator: 技术指标计算正常")
print("✅ FundamentalCalculator: 基本面评分正常")
print("✅ MacroCalculator: 宏观联动系数正常")
print("✅ DynamicPriceEngine: 三维综合计算正常")
print("✅ PriceValidator: 价格验证正常")
print("="*60)

cache.clear()
print("\n✅ 资源已清理")